In [0]:

# PHASE 11 - AUTOMATED PIPELINE HEALTH CHECK

from databricks.sdk import WorkspaceClient
from datetime import datetime, timezone

w = WorkspaceClient()

MAIN_JOB_NAME = "enterprise_ecommerce_dev_pipeline"

# Find the main processing job
jobs = [
    job for job in w.jobs.list()
    if job.settings
    and job.settings.name == MAIN_JOB_NAME
]

if len(jobs) != 1:
    raise RuntimeError(
        f"Expected one main job, found {len(jobs)}"
    )

job_id = jobs[0].job_id

# Retrieve recent completed executions
runs = list(
    w.jobs.list_runs(
        job_id=job_id,
        completed_only=True,
        limit=10
    )
)

if not runs:
    raise RuntimeError(
        "No completed pipeline runs found."
    )

# Select most recently started completed execution
latest = max(
    runs,
    key=lambda r: r.start_time or 0
)

state = latest.state

result = (
    state.result_state.value
    if state and state.result_state
    else "UNKNOWN"
)

print("PIPELINE HEALTH CHECK")
print("Job:", MAIN_JOB_NAME)
print("Job ID:", job_id)
print("Run ID:", latest.run_id)
print("Result:", result)

# Databricks Jobs API uses SUCCESS for successful runs
if result != "SUCCESS":
    raise RuntimeError(
        f"Pipeline health check failed: {result}"
    )

print("HEALTH CHECK PASSED")
